# 01 — Dataset Construction
Builds the set of adjacent utterance pairs from DailyDialog (Li et al., 2017; Hugging Face copy `anezatra/dailydialog`, train split). Only pairs whose previous utterance is a Question or a Directive (original DailyDialog label) are kept. The original act labels are used only to select and describe pairs; they are never shown to the annotating models.

Output: `dialogue_expectation_mismatch.csv` (3,000 pairs, label columns empty).

In [ ]:
!pip -q install datasets
import re
import pandas as pd
from datasets import load_dataset

## 1. Settings

In [ ]:
DATASET_NAME = "anezatra/dailydialog"        # Hugging Face copy of DailyDialog (Li et al., 2017)
MIN_PREV_WORDS, MIN_RESP_WORDS = 5, 3
MAX_WORDS, MAX_CONTEXT_WORDS = 40, 35
PER_CLASS = 2000            # candidate sample drawn per class (2 x 2,000 = 4,000 candidates)
FINAL_SIZE = 3000           # final dataset = first FINAL_SIZE rows of the shuffled candidate sample
SEED = 42

ACT_NAMES = {1: "Inform", 2: "Question", 3: "Directive", 4: "Commissive"}   # official DailyDialog coding
TARGET_ACTS = ["Question", "Directive"]

dataset = load_dataset(DATASET_NAME)
print(dataset)

## 2. Adjacent pairs
Each dialogue is parsed into utterances; an adjacent pair is (utterance *i*, utterance *i+1*) with utterance *i−1* as optional context. Dialogues whose utterance and label counts differ are skipped.

In [ ]:
def parse_dialog(value):
    """'["Hi .", "Hello !"]' (a string) -> ['Hi .', 'Hello !']"""
    value = value.strip()[1:-1]
    pattern = r'''(?:"((?:\\.|[^"])*)"|'((?:\\.|[^'])*)')'''
    out = []
    for dq, sq in re.findall(pattern, value, re.DOTALL):
        text = dq if dq else sq
        out.append(text.replace("\\'", "'").replace('\\"', '"').strip())
    return out

def parse_numbers(value):
    return [int(x) for x in re.findall(r"\d+", value)]

# quick self-test of the parser on a made-up example
_demo = parse_dialog("""["Can I help you ?", 'No , I\\'m just looking .']""")
assert _demo == ["Can I help you ?", "No , I'm just looking ."], _demo
print("Parser self-test passed.")

def create_pairs(split):
    pairs, skipped = [], 0
    for dialogue_id, ex in enumerate(split):
        utt, acts, emos = parse_dialog(ex["dialog"]), parse_numbers(ex["act"]), parse_numbers(ex["emotion"])
        if not (len(utt) == len(acts) == len(emos)):
            skipped += 1
            continue
        for i in range(len(utt) - 1):
            pairs.append({"dialogue_id": dialogue_id, "turn": i + 1,
                          "context": utt[i - 1] if i > 0 else "",
                          "previous_utterance": utt[i], "response": utt[i + 1],
                          "previous_act": acts[i], "response_act": acts[i + 1]})
    print(f"Dialogues skipped because lengths did not match: {skipped}")
    return pd.DataFrame(pairs)

pairs = create_pairs(dataset["train"])
pairs["previous_act_name"] = pairs["previous_act"].map(ACT_NAMES)
pairs["response_act_name"] = pairs["response_act"].map(ACT_NAMES)
print(f"All adjacent pairs in the training split: {len(pairs):,}")

## 3. Filtering
Every step is counted in the funnel table.

| Step | Rule |
|---|---|
| 1 | previous utterance is a Question or Directive |
| 2 | previous utterance 5–40 words, response 3–40 words |
| 3 | previous utterance is not a formulaic expression (greeting, thanks, ...) |
| 4 | previous utterance contains a surface cue of its act (regular expressions) |
| 5 | context at most 35 words |
| 6 | balanced sample, shuffled; first 3,000 pairs kept |

In [ ]:
WEAK = {"ok", "okay", "sure", "yes", "no", "thanks", "thank you", "all right", "alright", "hi", "hello",
        "hey", "good morning", "good afternoon", "good evening", "beg your pardon", "really", "no way"}

CUES = {
    "Question": [r"\?", r"\bwhat\b", r"\bwhere\b", r"\bwhen\b", r"\bwho\b", r"\bwhy\b", r"\bhow\b",
                 r"\bcan you\b", r"\bcould you\b", r"\bwould you\b", r"\bdo you\b", r"\bdid you\b",
                 r"\bare you\b", r"\bis it\b", r"\bhave you\b", r"\bwill you\b", r"\bmay i\b", r"\bcan i\b"],
    "Directive": [r"\bplease\b", r"\bcan you\b", r"\bcould you\b", r"\bwould you\b", r"\bwill you\b",
                  r"\bi need you to\b", r"\byou should\b", r"\byou must\b", r"\byou have to\b", r"\blet's\b"],
}

def n_words(s):
    return s.str.split().str.len()

def has_cue(row):
    return any(re.search(p, row["previous_utterance"].lower()) for p in CUES[row["previous_act_name"]])

funnel = [("All adjacent pairs (train split)", len(pairs))]
df = pairs.dropna(subset=["previous_utterance", "response"]).copy()

df = df[df["previous_act_name"].isin(TARGET_ACTS)].copy()
funnel.append(("1. Previous utterance is Question or Directive", len(df)))

df = df[(n_words(df["previous_utterance"]).between(MIN_PREV_WORDS, MAX_WORDS)) &
        (n_words(df["response"]).between(MIN_RESP_WORDS, MAX_WORDS))].copy()
funnel.append((f"2. Length limits ({MIN_PREV_WORDS}-{MAX_WORDS} words previous, {MIN_RESP_WORDS}-{MAX_WORDS} response)", len(df)))

clean = df["previous_utterance"].str.lower().str.strip().str.replace(r"[.!?,]+$", "", regex=True)
df = df[~clean.isin(WEAK)].copy()
funnel.append(("3. Not a formulaic expression", len(df)))

df = df[df.apply(has_cue, axis=1)].copy()
funnel.append(("4. Contains a cue of its act (regex)", len(df)))

df["context"] = df["context"].fillna("").astype(str)
df = df[(df["context"] == "") | (n_words(df["context"]) <= MAX_CONTEXT_WORDS)].copy()
funnel.append((f"5. Context ≤ {MAX_CONTEXT_WORDS} words (or none)", len(df)))

df["act_relation"] = (df["previous_act_name"] == df["response_act_name"]).map({True: "Same Act", False: "Different Act"})

funnel_df = pd.DataFrame(funnel, columns=["Step", "Pairs remaining"])
funnel_df["Removed"] = (-funnel_df["Pairs remaining"].diff()).fillna(0).astype(int)
print(funnel_df.to_string(index=False))
print("\nEligible pairs per class:\n", df["previous_act_name"].value_counts())

## 4. Sampling
Up to 2,000 pairs per act (Question, Directive), aiming for half with the same and half with a different original act in the response. The 4,000 candidates are shuffled (seed 42) and the first 3,000 are kept as the final dataset.

In [ ]:
parts = []
for act in TARGET_ACTS:
    subset = df[df["previous_act_name"] == act]
    same, diff = subset[subset["act_relation"] == "Same Act"], subset[subset["act_relation"] == "Different Act"]
    n_same = min(PER_CLASS // 2, len(same))
    n_diff = min(PER_CLASS - n_same, len(diff))
    if n_same and n_diff:
        parts.append(pd.concat([same.sample(n=n_same, random_state=SEED), diff.sample(n=n_diff, random_state=SEED)]))
    else:
        parts.append(subset.sample(n=min(PER_CLASS, len(subset)), random_state=SEED))

candidates = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
candidates.insert(0, "pair_id", range(1, len(candidates) + 1))
print(f"Candidate sample: {len(candidates):,} pairs")

# Final dataset = first FINAL_SIZE rows of the shuffled candidates (a random subset, pair_id 1..FINAL_SIZE)
sample = candidates.head(FINAL_SIZE).copy()
funnel.append((f"6. Balanced sample, shuffled, first {FINAL_SIZE:,} kept", len(sample)))
funnel_df = pd.DataFrame(funnel, columns=["Step", "Pairs remaining"])
funnel_df["Removed"] = (-funnel_df["Pairs remaining"].diff()).fillna(0).astype(int)

print(f"Final dataset size: {len(sample):,} pairs from {sample['dialogue_id'].nunique():,} dialogues")
print(pd.crosstab(sample["previous_act_name"], sample["act_relation"], margins=True))

## 5. Save

In [ ]:
cols = ["pair_id", "dialogue_id", "turn", "context", "previous_utterance", "response",
        "previous_act_name", "response_act_name", "act_relation"]
out = sample[cols].copy()
for c in ["Expected Response Function", "Actual Function", "Fulfillment", "Mismatch Type"]:
    out[c] = ""
out.to_csv("dialogue_expectation_mismatch.csv", index=False, encoding="utf-8-sig")
print("Saved dialogue_expectation_mismatch.csv")
# Optional check: the saved file should have exactly FINAL_SIZE rows, pair_id 1..FINAL_SIZE and four empty label columns
chk = pd.read_csv("dialogue_expectation_mismatch.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
assert len(chk) == FINAL_SIZE and chk["pair_id"].astype(int).tolist() == list(range(1, FINAL_SIZE + 1))
print(f"Check passed: {len(chk):,} rows.")